Pre-process ERA5 data to reduce size. Crop 2t and 2d to Caribbean and save in single nc to work folder.

In [1]:
import xarray as xr
import glob
from dask_jobqueue import PBSCluster
from dask.distributed import Client

# file check

In [18]:
ds = xr.open_dataset('/glade/campaign/cgd/cas/observations/ERA5/mon/t2m/era5.t2m.194001-202512.nc')
ds

<xarray.Dataset> Size: 4GB
Dimensions:     (valid_time: 1032, latitude: 721, longitude: 1440)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 8kB 1940-01-01 ... 2025-12-01
    expver      (valid_time) <U4 17kB ...
  * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    number      int64 8B ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 4GB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-02-09T06:48 GRIB to CDM+CF via cfgrib-0.9.1...

In [19]:
ds = xr.open_dataset('/gdex/data/d633000/e5.oper.an.sfc/194001/e5.oper.an.sfc.128_167_2t.ll025sc.1940010100_1940013123.nc')
ds

<xarray.Dataset> Size: 3GB
Dimensions:    (time: 744, latitude: 721, longitude: 1440)
Coordinates:
  * time       (time) datetime64[ns] 6kB 1940-01-01 ... 1940-01-31T23:00:00
  * latitude   (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude  (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
Data variables:
    VAR_2T     (time, latitude, longitude) float32 3GB ...
    utc_date   (time) int32 3kB ...
Attributes:
    DATA_SOURCE:          ECMWF: https://cds.climate.copernicus.eu, Copernicu...
    NETCDF_CONVERSION:    CISL RDA: Conversion from ECMWF GRIB1 data to netCDF4.
    NETCDF_VERSION:       4.8.1
    CONVERSION_PLATFORM:  Linux r1i7n2 4.12.14-95.51-default #1 SMP Fri Apr 1...
    CONVERSION_DATE:      Thu Mar 16 18:41:53 MDT 2023
    Conventions:          CF-1.6
    NETCDF_COMPRESSION:   NCO: Precision-preserving compression to netCDF4/HD...
    history:              Thu Mar 16 18:42:14 2023: ncks -4 --ppc default=7 e...
    NCO:                  netCDF Operators version 5.0.3 (Homepage = http://n...

In [16]:
ds = xr.open_dataset('/gdex/data/d633000/e5.oper.an.sfc/194108/e5.oper.an.sfc.128_168_2d.ll025sc.1941080100_1941083123.nc')
ds

<xarray.Dataset> Size: 3GB
Dimensions:    (time: 744, latitude: 721, longitude: 1440)
Coordinates:
  * time       (time) datetime64[ns] 6kB 1941-08-01 ... 1941-08-31T23:00:00
  * latitude   (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude  (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
Data variables:
    VAR_2D     (time, latitude, longitude) float32 3GB ...
    utc_date   (time) int32 3kB ...
Attributes:
    DATA_SOURCE:          ECMWF: https://cds.climate.copernicus.eu, Copernicu...
    NETCDF_CONVERSION:    CISL RDA: Conversion from ECMWF GRIB1 data to netCDF4.
    NETCDF_VERSION:       4.8.1
    CONVERSION_PLATFORM:  Linux r13i7n26 4.12.14-95.51-default #1 SMP Fri Apr...
    CONVERSION_DATE:      Tue May  9 16:37:31 MDT 2023
    Conventions:          CF-1.6
    NETCDF_COMPRESSION:   NCO: Precision-preserving compression to netCDF4/HD...
    history:              Tue May  9 16:37:48 2023: ncks -4 --ppc default=7 e...
    NCO:                  netCDF Operators version 5.0.3 (Homepage = http://n...

# file selection

In [2]:
t_files = glob.glob('/gdex/data/d633000/e5.oper.an.sfc/*/*2t*.nc')
t_files

['/gdex/data/d633000/e5.oper.an.sfc/202207/e5.oper.an.sfc.128_167_2t.ll025sc.2022070100_2022073123.nc',
 '/gdex/data/d633000/e5.oper.an.sfc/198908/e5.oper.an.sfc.128_167_2t.ll025sc.1989080100_1989083123.nc',
 '/gdex/data/d633000/e5.oper.an.sfc/199204/e5.oper.an.sfc.128_167_2t.ll025sc.1992040100_1992043023.nc',
 '/gdex/data/d633000/e5.oper.an.sfc/201904/e5.oper.an.sfc.128_167_2t.ll025sc.2019040100_2019043023.nc',
 '/gdex/data/d633000/e5.oper.an.sfc/194402/e5.oper.an.sfc.128_167_2t.ll025sc.1944020100_1944022923.nc',
 '/gdex/data/d633000/e5.oper.an.sfc/198004/e5.oper.an.sfc.128_167_2t.ll025sc.1980040100_1980043023.nc',
 '/gdex/data/d633000/e5.oper.an.sfc/200111/e5.oper.an.sfc.128_167_2t.ll025sc.2001110100_2001113023.nc',
 '/gdex/data/d633000/e5.oper.an.sfc/198601/e5.oper.an.sfc.128_167_2t.ll025sc.1986010100_1986013123.nc',
 '/gdex/data/d633000/e5.oper.an.sfc/198807/e5.oper.an.sfc.128_167_2t.ll025sc.1988070100_1988073123.nc',
 '/gdex/data/d633000/e5.oper.an.sfc/198411/e5.oper.an.sfc.128_16

In [3]:
d_files = glob.glob('/gdex/data/d633000/e5.oper.an.sfc/*/*2d*.nc')
d_files

['/gdex/data/d633000/e5.oper.an.sfc/202207/e5.oper.an.sfc.128_168_2d.ll025sc.2022070100_2022073123.nc',
 '/gdex/data/d633000/e5.oper.an.sfc/198908/e5.oper.an.sfc.128_168_2d.ll025sc.1989080100_1989083123.nc',
 '/gdex/data/d633000/e5.oper.an.sfc/199204/e5.oper.an.sfc.128_168_2d.ll025sc.1992040100_1992043023.nc',
 '/gdex/data/d633000/e5.oper.an.sfc/201904/e5.oper.an.sfc.128_168_2d.ll025sc.2019040100_2019043023.nc',
 '/gdex/data/d633000/e5.oper.an.sfc/194402/e5.oper.an.sfc.128_168_2d.ll025sc.1944020100_1944022923.nc',
 '/gdex/data/d633000/e5.oper.an.sfc/198004/e5.oper.an.sfc.128_168_2d.ll025sc.1980040100_1980043023.nc',
 '/gdex/data/d633000/e5.oper.an.sfc/200111/e5.oper.an.sfc.128_168_2d.ll025sc.2001110100_2001113023.nc',
 '/gdex/data/d633000/e5.oper.an.sfc/198601/e5.oper.an.sfc.128_168_2d.ll025sc.1986010100_1986013123.nc',
 '/gdex/data/d633000/e5.oper.an.sfc/198807/e5.oper.an.sfc.128_168_2d.ll025sc.1988070100_1988073123.nc',
 '/gdex/data/d633000/e5.oper.an.sfc/198411/e5.oper.an.sfc.128_16

# dask setup

In [4]:
cluster = PBSCluster(
    cores=4,
    memory='64GB',
    processes=1,
    queue='casper',
    local_directory='$TMPDIR',
    account='P93300313',
    walltime='4:00:00'
)
cluster.scale(jobs=4)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.100:45115,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


2026-05-27 13:04:21,284 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='jupyterhub.hpc.ucar.edu', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/tornado/websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
  File "/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/tornado/web.py", line 3409, in wrapper
    return method(self, *args, **kwargs)
  File "/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expiration if necessary")
bokeh.protocol.exceptions.ProtocolError: Token is expired. Configure the 

# get data and crop

In [18]:
# I have not found a way to add a variable to preprocess function, 
# doc mentions it takes a ds type but no info on adding other arguments,
# end up with a function per veriable.
def crop_t_ds(ds):
    ds = ds['VAR_2T']
    ds.coords['longitude'] = (ds.coords['longitude'] + 180) % 360 - 180 # easier to use east west when slicing, lat already in N-S
    ds = ds.sortby(ds.longitude)
    ds_crop = ds.sel(latitude=slice(10, 25), longitude=slice(-85, -60))
    return ds_crop

In [19]:
T = xr.open_mfdataset(t_files, combine='nested',
                     engine='h5netcdf', parallel=True,
                     concat_dim=['time'], preprocess=crop_t_ds)

In [23]:
def crop_d_ds(ds):
    ds = ds['VAR_2D']
    ds.coords['longitude'] = (ds.coords['longitude'] + 180) % 360 - 180
    ds = ds.sortby(ds.longitude)
    ds_crop = ds.sel(latitude=slice(10,25), longitude=slice(-85, -60))
    return ds_crop

In [24]:
D = xr.open_mfdataset(d_files, combine='nested',
                     engine='h5netcdf',parallel=True,
                     concat_dim=['time'], preprocess=crop_d_ds)

# output to reusable file

In [ ]:
T.to_zarr('/glade/work/acruz/Caribbean_Heat_data/ERA5/sfc_hourly_temp', mode='w', consolidated=True)

/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 295.58 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [ ]:
D.to_zarr('/glade/work/acruz/Caribbean_Heat_data/ERA5/sfc_hourly_dew', mode='w', consolidated=True)

# shutdown dask

In [28]:
client.shutdown()